In [99]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy.stats import entropy
from scipy import signal
from statsmodels.tsa import tsatools, seasonal
from collections import defaultdict
import os, sys
from tqdm import tqdm
from typing import Literal
from meegkit.detrend import detrend as meeg_detrend, regress
from sklearn.decomposition import FastICA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from umap import UMAP

import matplotlib.pyplot as plt
import neurokit2

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
import extractor
import wavelets

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
os.chdir(r'J:\Onderzoek\21-763_rvanes_MiniECG-2-Data\E_ResearchData\2_ResearchData\numpy\numpy')

In [ ]:
FACTOR = 1e-3
NUM_ICA_COMPONENTS = 4

In [ ]:
def DeTrender(ts: np.ndarray,
              how: Literal=['poly', 'MSTL', 'meeg'],
              order=1, 
              period=100, 
              windows=101, 
              cutsize=250,
              median_window=20,
              iterate=5):
    if how == 'poly':
        return tsatools.detrend(ts, order=order)
    elif how == 'poly_np':
        z = np.polyfit(np.arange(0,ts.shape[0],1), ts, order)
        y_poly = np.poly1d(z)
        return ts-y_poly(np.arange(0,ts.shape[0],1))
    elif how == 'MSTL':
        detrender= seasonal.MSTL(ts, periods=period, windows=windows, iterate=iterate).fit()
        return detrender.seasonal
    elif how == 'meeg':
        '''
         Use sinusoids as basis functions:
          This will suffer from the Gibbs-phenomenon. 
          To mitigate this, we have to cut the head/tail
        '''
        _ts = np.array(ts).astype(float)
        return meeg_detrend(_ts, order=order, basis="sinusoids")[0][cutsize:-cutsize] 
    elif how == 'sliding_median':
        '''
         Make base plot using sliding mean, subtract from raw 
        '''
        sl= np.median(np.lib.stride_tricks.sliding_window_view(ts, (median_window,)), axis=1)
        return ts[median_window-1:] - sl
    elif how == 'sliding_mean':
        '''
         Make base plot using sliding mean, subtract from raw 
        '''
        sl= np.mean(np.lib.stride_tricks.sliding_window_view(ts, (median_window,)), axis=1)
        return ts[median_window-1:] - sl
    else:
        raise ValueError(f"For now we only accept ['poly', 'MSTL', 'meeg]")
    
def HighPassFilter(ts):
    pass

def LowPassFilter(ts):
    pass

class ButterworthBandpassFilter:
    def __init__(self, lowcut, highcut, fs, order=5):
        self.lowcut = lowcut
        self.highcut = highcut
        self.fs = fs
        self.order = order
        self.b, self.a = self._design_filter()

    def _design_filter(self):
        nyq = 0.5 * self.fs
        low = self.lowcut / nyq
        high = self.highcut / nyq
        return signal.butter(self.order, [low, high], btype='band')

    def apply_filter(self, data):
        return signal.lfilter(self.b, self.a, data)

class ICASignalSeparator:
    def __init__(self, n_components=None, max_iter=250):
        self.n_components = n_components
        self.ica = FastICA(n_components=n_components, random_state=42, max_iter=max_iter)
        
    def separate(self, mixed_signals):
        """
        Separate mixed signals using ICA.
        
        :param mixed_signals: 2D array, shape (n_samples, n_features)
        :return: 2D array of separated signals
        """
        return self.ica.fit_transform(mixed_signals.T).T
    
    def plot_signals(self, original_signals, mixed_signals, separated_signals):
        n_signals = original_signals.shape[0]
        time = np.linspace(0, 10, original_signals.shape[1])
        
        fig, axs = plt.subplots(3, n_signals, figsize=(15, 10))
        fig.suptitle('Original, Mixed, and Separated Signals')
        
        for i in range(n_signals):
            axs[0, i].plot(time, original_signals[i])
            axs[0, i].set_title(f'Original Signal {i+1}')
            
            axs[1, i].plot(time, mixed_signals[i])
            axs[1, i].set_title(f'Mixed Signal {i+1}')
            
            axs[2, i].plot(time, separated_signals[i])
            axs[2, i].set_title(f'Separated Signal {i+1}')
        
        for ax in axs.flat:
            ax.set(xlabel='Time', ylabel='Amplitude')
        
        plt.tight_layout()
        plt.show()


'''
Ideally we have a signal-processing pipe that maximally consists of 
ts -> Interpolator -> SignalSeparator -> DeTrender -> HighPassFilter -> LowPassFilter

'''

In [ ]:
# LOAD IN ALL THE NPYs
DIRS = os.listdir('.')
ECG_DICT = defaultdict(list)
        if FILE.endswith('.npy'):
            arr = np.load(os.path.join(DIR, FILE))
            vars = np.var(arr, axis=1)
            if arr.shape[1] > 0:
                if any(vars == 0) == False:
                    ECG_DICT[DIR].append(arr)
                else:
                    SKIPPED.append(f"zero variance band {DIR}/{FILE}")
            else:
                SKIPPED.append(f"empty {DIR}/{FILE}")

In [ ]:
c = 0
for k,v in ECG_DICT.items():
    c += len(v)
print(f"We kept {c} ECG results, skipping {len(SKIPPED)} because they had at least one flat band or were empty..")

# Viz

In [ ]:
ECG_DICT.keys()

In [ ]:
raw = ECG_DICT['20230808'][0]*FACTOR

In [ ]:
separator = ICASignalSeparator(n_components=NUM_ICA_COMPONENTS, max_iter=2000)
separated = separator.separate(raw)

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=2, figsize=(30,20))

ax[0,0].plot(raw[0])
ax[1,0].plot(raw[1])
ax[2,0].plot(raw[2])
ax[3,0].plot(raw[3])
ax[4,0].plot(raw[4])
ax[5,0].plot(raw[5])
ax[6,0].plot(raw[6])
ax[7,0].plot(raw[7])

for i in range(0, NUM_ICA_COMPONENTS):
    ax[i,1].plot(separated[i])

In [ ]:
from pydmd import DMD
from pydmd.plotter import plot_summary


In [ ]:
dmd = DMD(svd_rank=2)
dmd.fit(raw.T)

In [ ]:
plot_summary(dmd)

In [ ]:
separated_dmd = np.real(dmd.modes).T

In [ ]:
detrended_list = []
banded_list = []
for i, _signal in enumerate(raw):
    detrended = DeTrender(_signal, how='sliding_median', order=3, cutsize=250, median_window=5)
    banded = ButterworthBandpassFilter(0.67, 100, 250, order=5).apply_filter(detrended)
    detrended_list.append(detrended)
    banded_list.append(banded)
    

In [ ]:
num_plots = len(banded_list)
fig, ax = plt.subplots(figsize=(18, 20), ncols=1, nrows=num_plots)
for i in range(num_plots):
    ax[i].plot(detrended_list[i], label='detrended')
    ax[i].plot(banded_list[i], alpha=0.45, label='band filtered')
    ax[i].legend()

For an ECG feature extractor it makes sense to use **multiple** bandpass filters, **multiple** detrending methods and **multiple** ICAs/DMDs.

## Apply TimEx extractor

In [ ]:
import pycatch22

In [ ]:
# OPTION 1 
# go through the raw series and apply catch22, plus the first N Fourier coefficients

extracted_results = []
for k,v in tqdm(ECG_DICT.items()):
    for i, _signal in enumerate(v):
        for j, channel in enumerate(_signal):
            detrended = DeTrender(channel*FACTOR, how='sliding_median', order=24, cutsize=250)
            banded = ButterworthBandpassFilter(0.5,40, 250, order=5).apply_filter(detrended)
            
            wvc = wavelets.extract_wavelet_features(channel, wavelet='db4', level=3, num_features=6)
            fcs = extractor.extract_fft_features(channel, num_features=8, max_frequency=40)
            c22s = pycatch22.catch22_all(channel)
            ##
            out_dict = {'date': k, 'signal': i, 'channel': j}
            out_dict.update(wvc)
            out_dict.update(fcs)
            out_dict.update(dict(zip(c22s['names'], c22s['values'])))
            extracted_results.append(out_dict)
extracted_df = pd.DataFrame(extracted_results)    

  8%|▊         | 39/515 [01:57<32:59,  4.16s/it] 

In [ ]:
extracted_df_channel_0 = extracted_df.loc[extracted_df.channel==0].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_1 = extracted_df.loc[extracted_df.channel==1].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_2 = extracted_df.loc[extracted_df.channel==2].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_3 = extracted_df.loc[extracted_df.channel==3].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_4 = extracted_df.loc[extracted_df.channel==4].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_5 = extracted_df.loc[extracted_df.channel==5].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_6 = extracted_df.loc[extracted_df.channel==6].drop('channel', axis=1).set_index(['date', 'signal'])
extracted_df_channel_7 = extracted_df.loc[extracted_df.channel==7].drop('channel', axis=1).set_index(['date', 'signal'])

In [ ]:
umapper = UMAP(n_components = 3, n_neighbors=50, densmap=True)
scaler = StandardScaler()
le_pipe = Pipeline([('scaler', scaler),
                    ('mapper', umapper)])

In [ ]:
channel_0_umap = le_pipe.fit_transform(extracted_df_channel_0)
channel_1_umap = le_pipe.fit_transform(extracted_df_channel_1)
channel_2_umap = le_pipe.fit_transform(extracted_df_channel_2)
channel_3_umap = le_pipe.fit_transform(extracted_df_channel_3)
channel_4_umap = le_pipe.fit_transform(extracted_df_channel_4)
channel_5_umap = le_pipe.fit_transform(extracted_df_channel_5)
channel_6_umap = le_pipe.fit_transform(extracted_df_channel_6)
channel_7_umap = le_pipe.fit_transform(extracted_df_channel_7)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=4, figsize=(18,20))

ax[0,0].scatter(channel_0_umap[:,0], channel_0_umap[:,1], alpha=0.05)
ax[1,0].scatter(channel_1_umap[:,0], channel_1_umap[:,1], alpha=0.05)
ax[2,0].scatter(channel_2_umap[:,0], channel_2_umap[:,1], alpha=0.05)
ax[3,0].scatter(channel_3_umap[:,0], channel_3_umap[:,1], alpha=0.05)

#ax[0,0].scatter(channel_0_umap[:,0], channel_0_umap[:,2], alpha=0.05)
#ax[1,0].scatter(channel_1_umap[:,0], channel_1_umap[:,2], alpha=0.05)
#ax[2,0].scatter(channel_2_umap[:,0], channel_2_umap[:,2], alpha=0.05)
#ax[3,0].scatter(channel_3_umap[:,0], channel_3_umap[:,2], alpha=0.05)

# ax[0,0].scatter(channel_0_umap[:,1], channel_0_umap[:,2], alpha=0.05)
# ax[1,0].scatter(channel_1_umap[:,1], channel_1_umap[:,2], alpha=0.05)
# ax[2,0].scatter(channel_2_umap[:,1], channel_2_umap[:,2], alpha=0.05)
# ax[3,0].scatter(channel_3_umap[:,1], channel_3_umap[:,2], alpha=0.05)

########################################################################

ax[0,1].scatter(channel_4_umap[:,0], channel_4_umap[:,1], alpha=0.05)
ax[1,1].scatter(channel_5_umap[:,0], channel_5_umap[:,1], alpha=0.05)
ax[2,1].scatter(channel_6_umap[:,0], channel_6_umap[:,1], alpha=0.05)
ax[3,1].scatter(channel_7_umap[:,0], channel_7_umap[:,1], alpha=0.05)

#ax[0,1].scatter(channel_4_umap[:,0], channel_4_umap[:,2], alpha=0.05)
#ax[1,1].scatter(channel_5_umap[:,0], channel_5_umap[:,2], alpha=0.05)
#ax[2,1].scatter(channel_6_umap[:,0], channel_6_umap[:,2], alpha=0.05)
#ax[3,1].scatter(channel_7_umap[:,0], channel_7_umap[:,2], alpha=0.05)

# ax[0,1].scatter(channel_4_umap[:,1], channel_4_umap[:,2], alpha=0.05)
# ax[1,1].scatter(channel_5_umap[:,1], channel_5_umap[:,2], alpha=0.05)
# ax[2,1].scatter(channel_6_umap[:,1], channel_6_umap[:,2], alpha=0.05)
# ax[3,1].scatter(channel_7_umap[:,1], channel_7_umap[:,2], alpha=0.05)

In [ ]:
# OPTION 2 APPEND AS SEPARATE SERIES: Go through all the waveforms of length M
# Extract modes with DMD (gives D series) and ICA (gives I series), append to original (of N channels) -> gives D+I+N channels 
# THEN
# detrend with median and meeg -> 2X (D+I+N) channels
# THEN
# ADD bandpass filter with -> 4X (D+I+N) channels


# OPTION 3 APPEND  TO SERIES: Go through all the waveforms of length M
# Extract modes with DMD (gives D series) and ICA (gives I series), append to original (of N channels) -> gives D+I+N channels 
# THEN
# detrend with median and meeg -> (D+I+N) channels of length 2 X M 
# THEN
# ADD bandpass filter with -> (D+I+N) channels of length 3 X M 
## COMPLEXITY HERE is that we need to smoothly connect the waveforms


# Peak/valley characteristics

In [ ]:
def peak_valley_coefficients(ts):
    '''
        Function to extract 
            the number of peaks/valley
            the average value of the peaks/valleys
            the max/min
            the average time between peaks/valleys/peak-valleys
    '''
    pass